<a href="https://colab.research.google.com/github/mayurivedpathak01-alt/CodeVedX-Data-Science-Internship/blob/main/Movie_Rating_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if "IMDb" in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/IMDb%20Movies%20India (1).csv


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/content/drive/MyDrive/IMDb%20Movies%20India (1).csv
/content/drive/MyDrive/Colab Notebooks/Advertising.csv


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'IMDb' in file or 'imdb' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/IMDb%20Movies%20India (1).csv


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.lower().endswith('.csv'):
            print(file)
            print(os.path.join(root, file))
            print("----------------")

IMDb%20Movies%20India (1).csv
/content/drive/MyDrive/IMDb%20Movies%20India (1).csv
----------------
Advertising.csv
/content/drive/MyDrive/Colab Notebooks/Advertising.csv
----------------


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/IMDb%20Movies%20India (1).csv', encoding='latin1')

print(df.head())

                                 Name    Year Duration            Genre  \
0                                         NaN      NaN            Drama   
1  #Gadhvi (He thought he was Gandhi)  (2019)  109 min            Drama   
2                         #Homecoming  (2021)   90 min   Drama, Musical   
3                             #Yaaram  (2019)  110 min  Comedy, Romance   
4                   ...And Once Again  (2010)  105 min            Drama   

   Rating Votes            Director       Actor 1             Actor 2  \
0     NaN   NaN       J.S. Randhawa      Manmauji              Birbal   
1     7.0     8       Gaurav Bakshi  Rasika Dugal      Vivek Ghamande   
2     NaN   NaN  Soumyajit Majumdar  Sayani Gupta   Plabita Borthakur   
3     4.4    35          Ovais Khan       Prateik          Ishita Raj   
4     NaN   NaN        Amol Palekar  Rajat Kapoor  Rituparna Sengupta   

           Actor 3  
0  Rajendra Bhatia  
1    Arvind Jangid  
2       Roy Angana  
3  Siddhant Kapoor  
4    

In [ ]:
print(df.shape)

print(df.columns)

df.head()

df.info()

df.isnull().sum()

(15509, 10)
Index(['Name', 'Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director',
       'Actor 1', 'Actor 2', 'Actor 3'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  object 
 2   Duration  7240 non-null   object 
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   object 
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(1), object(9)
memory usage: 1.2+ MB


,0
Name,0
Year,528
Duration,8269
Genre,1877
Rating,7590
Votes,7589
Director,525
Actor 1,1617
Actor 2,2384
Actor 3,3144


In [ ]:
# Remove rows where Rating is missing
df = df.dropna(subset=['Rating'])

# Clean Votes column
df['Votes'] = df['Votes'].astype(str).str.replace(',', '', regex=False)
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')

# Clean Year column
df['Year'] = df['Year'].astype(str).str.extract('(\d{4})')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# Clean Duration column
df['Duration'] = df['Duration'].astype(str).str.extract('(\d+)')
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')

print(df.head())

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:13: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:13: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_961/2287448388.py:9: SyntaxWarning: invalid escape sequence '\d'
  df['Year'] = df['Year'].astype(str).str.extract('(\d{4})')
/tmp/ipykernel_961/2287448388.py:13: SyntaxWarning: invalid escape sequence '\d'
  df['Duration'] = df['Duration'].astype(str).str.extract('(\d+)')


                                 Name  Year  Duration  \
1  #Gadhvi (He thought he was Gandhi)  2019     109.0   
3                             #Yaaram  2019     110.0   
5                ...Aur Pyaar Ho Gaya  1997     147.0   
6                           ...Yahaan  2005     142.0   
8                  ?: A Question Mark  2012      82.0   

                       Genre  Rating  Votes        Director          Actor 1  \
1                      Drama     7.0      8   Gaurav Bakshi     Rasika Dugal   
3            Comedy, Romance     4.4     35      Ovais Khan          Prateik   
5     Comedy, Drama, Musical     4.7    827    Rahul Rawail       Bobby Deol   
6        Drama, Romance, War     7.4   1086  Shoojit Sircar  Jimmy Sheirgill   
8  Horror, Mystery, Thriller     5.6    326   Allyson Patel        Yash Dave   

                  Actor 2          Actor 3  
1          Vivek Ghamande    Arvind Jangid  
3              Ishita Raj  Siddhant Kapoor  
5  Aishwarya Rai Bachchan    Shammi Kapoo

In [ ]:
features = [
    "Year",
    "Duration",
    "Votes",
    "Genre",
    "Director",
    "Actor 1",
    "Actor 2",
    "Actor 3"
]

X = df[features]
y = df["Rating"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ["Genre", "Director", "Actor 1", "Actor 2", "Actor 3"]

for col in categorical_cols:
    X[col] = X[col].fillna("Unknown")
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

for col in ["Year", "Duration", "Votes"]:
    X[col] = X[col].fillna(X[col].median())

/tmp/ipykernel_961/3019797066.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna("Unknown")
/tmp/ipykernel_961/3019797066.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col].astype(str))
/tmp/ipykernel_961/3019797066.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully!")

Model Trained Successfully!


In [ ]:
pred = model.predict(X_test)

print("Predicted Ratings:")
print(pred[:10])

Predicted Ratings:
[4.4375 5.2435 4.858  5.666  5.3435 5.916  5.7175 6.9575 5.672  5.9295]


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)

Mean Absolute Error (MAE): 0.8298238636363635
Root Mean Squared Error (RMSE): 1.0995039974479586
R² Score: 0.3497500469564032


In [ ]:
results = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating": pred
})

print(results.head(20))

    Actual Rating  Predicted Rating
0             3.3            4.4375
1             5.3            5.2435
2             5.7            4.8580
3             7.2            5.6660
4             3.5            5.3435
5             7.2            5.9160
6             3.8            5.7175
7             6.9            6.9575
8             5.2            5.6720
9             7.4            5.9295
10            4.3            6.1935
11            7.2            6.7185
12            5.6            4.8565
13            3.9            4.7255
14            3.2            4.7840
15            6.4            5.0950
16            5.8            5.1220
17            6.4            6.0550
18            7.6            7.2180
19            2.9            5.6420


In [ ]:
results.to_csv("movie_rating_predictions.csv", index=False)

print("Results saved successfully!")

Results saved successfully!


In [ ]:
results = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating": pred
})

print(results.head(20))

    Actual Rating  Predicted Rating
0             3.3            4.4375
1             5.3            5.2435
2             5.7            4.8580
3             7.2            5.6660
4             3.5            5.3435
5             7.2            5.9160
6             3.8            5.7175
7             6.9            6.9575
8             5.2            5.6720
9             7.4            5.9295
10            4.3            6.1935
11            7.2            6.7185
12            5.6            4.8565
13            3.9            4.7255
14            3.2            4.7840
15            6.4            5.0950
16            5.8            5.1220
17            6.4            6.0550
18            7.6            7.2180
19            2.9            5.6420


In this project, I built a Movie Rating Prediction model using the IMDb India Movies dataset. First, I cleaned and prepared the data by handling missing values and converting the required columns. Then, I trained a Random Forest Regression model to predict movie ratings. Finally, I evaluated the model using MAE, RMSE, and R² Score. This project helped me understand data preprocessing, model training, prediction, and model evaluation using machine learning. Overall, the model gave good results and improved my practical knowledge of data science.